# 19. Scaling Analysis（spec §19）

## 学習目標
- 同一条件（コーパス・トークン数・アーキ・optimizer）で**サイズだけ**を変えたときの val loss を見る
- パラメータ数と性能・計算コストの trade-off を理解する
- 「大きいほど良い」の限界と前提（同一トークン予算）を意識する

In [1]:
import warnings
warnings.filterwarnings("ignore")
import json, math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "plotly_mimetype"
from jp_llm_lab.utils.io import repo_root, load_json, read_jsonl
ROOT = repo_root()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
M2_RUN = ROOT / "experiments/runs/m2_model_m_classical_seed42"
L_RUN = ROOT / "experiments/runs/m4_model_l_modern_seed42"
M5 = ROOT / "experiments/analysis_m5"

In [2]:
from jp_llm_lab.visualization.comparison import size_scaling_figure
scaling = load_json(ROOT / "experiments/comparisons/scaling.json")
pts = scaling["points"]
size_scaling_figure(pts).show()
pd.DataFrame(pts)[["name","n_params","val_loss","ppl","wallclock_sec"]]

,name,n_params,val_loss,ppl,wallclock_sec
0,xs,1852544,6.236089,510.856756,9.1
1,s,6769920,5.646111,283.187926,13.4
2,m,13767552,5.317512,203.876038,19.0
3,l,29499904,5.582968,265.859595,34.5


In [3]:
# パラメータ vs val loss。単調か？ 最小 loss のサイズは？
import numpy as np
best = min(pts, key=lambda p: p["val_loss"])
print(f"最小 val_loss は {best['name']}（{best['n_params']:,} params, val {best['val_loss']:.3f}）")
mono = all(pts[i]["val_loss"] >= pts[i+1]["val_loss"] for i in range(len(pts)-1))
print(f"単調減少か？ {mono}")
print(f"総トークン予算 {scaling['tokens_budget']:,}（全サイズ同一）— 大きいモデルほど token/param が不足")
for p in pts:
    print(f'  {p["name"]:3s} {p["n_params"]:>10,} params  val {p["val_loss"]:.3f}  '
          f'tokens/param={scaling["tokens_budget"]/p["n_params"]:.2f}')

最小 val_loss は m（13,767,552 params, val 5.318）
単調減少か？ False
総トークン予算 6,144,000（全サイズ同一）— 大きいモデルほど token/param が不足
  xs   1,852,544 params  val 6.236  tokens/param=3.32
  s    6,769,920 params  val 5.646  tokens/param=0.91
  m   13,767,552 params  val 5.318  tokens/param=0.45
  l   29,499,904 params  val 5.583  tokens/param=0.21


## Observation / Interpretation / Caveat
- **Observation（実測・非単調）**: xs→s→m では val loss が下がる（6.24→5.65→5.32）が、**最大の l（30M）は m（13.7M）より悪化**（上のセルが実測）。つまり「大きいほど良い」は**この固定トークン予算では成り立たなかった**。
- **Interpretation**: 全サイズ同じ ~6M トークンしか与えていないため、最大モデルは **token/param が不足（データ枯渇）** して十分に学習できていない。これは Chinchilla 的な「計算最適」の核心 — パラメータを増やすなら**それに見合うトークン数**が要る。実際 M4 の Model L は同じ構成を 100M トークン（16倍）で学習し val 3.51 に到達しており、データを与えれば大モデルが勝つことを別途示している。
- **Caveat**: 固定トークン予算・小規模・単一シード・500 step。最適トークン数調整（サイズごとに変える）はしていないので、これは「compute-optimal 曲線」ではなく「固定予算での断面」。100M+ への外挿不可。

## 確認問題
1. なぜ最大モデルが最良でないのか。「容量」と「トークン数」のどちらが律速か。
2. Model L が M4 で val 3.51 に到達できたのに、この曲線で l が 5.58 なのはなぜか。
3. 「大きいほど良い」を正しく主張するには、独立変数に何を加える必要があるか。

**次へ**: [20_probability_calibration](20_probability_calibration.ipynb)